# 傅里叶变换演示

本 Notebook 演示如何使用 NumPy 和 Matplotlib 进行傅里叶变换分析，包括：
- 生成合成信号（正弦波 + 余弦波 + 噪声）
- 计算快速傅里叶变换（FFT）
- 绘制时域图和频域幅度谱
- 逆变换恢复原始信号

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 1. 生成合成信号

In [ ]:
np.random.seed(42)

sampling_freq = 1000  # 采样频率 (Hz)
duration = 2          # 信号持续时间 (秒)
t = np.linspace(0, duration, int(sampling_freq * duration), endpoint=False)

freq1 = 5    # 第一个频率分量 (Hz)
freq2 = 15   # 第二个频率分量 (Hz)
freq3 = 30   # 第三个频率分量 (Hz)

signal = (np.sin(2 * np.pi * freq1 * t) + 
          0.5 * np.cos(2 * np.pi * freq2 * t) + 
          0.3 * np.sin(2 * np.pi * freq3 * t))

noise_amplitude = 0.2
noise = noise_amplitude * np.random.randn(len(t))
signal_with_noise = signal + noise

print(f"信号长度: {len(signal_with_noise)} 个采样点")
print(f"时间范围: 0 到 {duration} 秒")

## 2. 计算快速傅里叶变换 (FFT)

In [ ]:
n = len(signal_with_noise)
fft_result = np.fft.fft(signal_with_noise)
frequencies = np.fft.fftfreq(n, 1/sampling_freq)

magnitude_spectrum = np.abs(fft_result) / n
magnitude_spectrum = magnitude_spectrum[:n//2]
positive_frequencies = frequencies[:n//2]

## 3. 绘制时域图

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(t, signal, label='原始无噪声信号', color='blue', alpha=0.7)
plt.xlabel('时间 (秒)')
plt.ylabel('幅度')
plt.title('原始无噪声信号')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
plt.plot(t, signal_with_noise, label='含噪声的合成信号', color='red', alpha=0.7)
plt.xlabel('时间 (秒)')
plt.ylabel('幅度')
plt.title('含噪声的合成信号')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 绘制频域幅度谱

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(positive_frequencies, 2 * magnitude_spectrum, color='green')
plt.xlabel('频率 (Hz)')
plt.ylabel('幅度')
plt.title('频域幅度谱')
plt.xlim(0, 50)  # 只显示前50Hz
plt.grid(True, alpha=0.3)

for freq in [freq1, freq2, freq3]:
    plt.axvline(x=freq, color='red', linestyle='--', alpha=0.5, label=f'{freq} Hz')

plt.legend()
plt.show()

## 5. 逆傅里叶变换恢复信号

In [ ]:
reconstructed_signal = np.fft.ifft(fft_result)
reconstructed_signal = np.real(reconstructed_signal)

In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(t, signal_with_noise, label='原始含噪声信号', color='red', alpha=0.7)
plt.xlabel('时间 (秒)')
plt.ylabel('幅度')
plt.title('原始含噪声信号')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 2)
plt.plot(t, reconstructed_signal, label='逆变换恢复信号', color='blue', alpha=0.7)
plt.xlabel('时间 (秒)')
plt.ylabel('幅度')
plt.title('逆傅里叶变换恢复信号')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 3)
difference = signal_with_noise - reconstructed_signal
plt.plot(t, difference, label='差值', color='purple', alpha=0.7)
plt.xlabel('时间 (秒)')
plt.ylabel('幅度')
plt.title('原始信号与恢复信号的差值')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
max_difference = np.max(np.abs(difference))
print(f"原始信号与恢复信号的最大差值: {max_difference:.2e}")
print(f"均方误差 (MSE): {np.mean(difference**2):.2e}")

## 6. 频域滤波演示

In [ ]:
filtered_fft = fft_result.copy()
cutoff_freq = 40
filtered_fft[np.abs(frequencies) > cutoff_freq] = 0

filtered_signal = np.fft.ifft(filtered_fft)
filtered_signal = np.real(filtered_signal)

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(t, signal_with_noise, label='含噪声信号', color='red', alpha=0.5)
plt.plot(t, signal, label='原始无噪声信号', color='blue', alpha=0.7)
plt.xlabel('时间 (秒)')
plt.ylabel('幅度')
plt.title('原始信号对比')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
plt.plot(t, filtered_signal, label='滤波后信号', color='green', alpha=0.7)
plt.plot(t, signal, label='原始无噪声信号', color='blue', alpha=0.5, linestyle='--')
plt.xlabel('时间 (秒)')
plt.ylabel('幅度')
plt.title(f'低通滤波后信号 (截止频率: {cutoff_freq} Hz)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 总结

本 Notebook 演示了：
1. **信号生成**：创建了包含三个频率分量（5Hz、15Hz、30Hz）和噪声的合成信号
2. **FFT 计算**：使用 NumPy 的 FFT 函数将时域信号转换到频域
3. **频谱分析**：绘制了幅度谱，清晰地显示了信号中的频率成分
4. **逆变换**：通过逆 FFT 成功恢复了原始信号
5. **频域滤波**：展示了如何在频域进行低通滤波来去除噪声

傅里叶变换是信号处理的核心工具，它使我们能够在频域分析和处理信号！